### Read Bronze Volumes

In [0]:
raw_path = "/Volumes/fintech/bronze/raw_files/"

df_raw = (
    spark.read
    .format("json")
    .option("multiLine", True)
    .load(raw_path)
)

display(df_raw)

In [0]:
df_raw.printSchema()

### Apply Schema To Data

In [0]:
from pyspark.sql.functions import (
    col,
    current_timestamp,
    to_date,
)


df_bronze = (
    df_raw
    .select(
        col("symbol")
            .cast("string")
            .alias("symbol"),

        to_date(col("date"))
            .alias("trade_date"),

        col("open")
            .cast("decimal(18,4)")
            .alias("open"),

        col("high")
            .cast("decimal(18,4)")
            .alias("high"),

        col("low")
            .cast("decimal(18,4)")
            .alias("low"),

        col("close")
            .cast("decimal(18,4)")
            .alias("close"),

        col("volume")
            .cast("long")
            .alias("volume"),

        col("change")
            .cast("decimal(18,4)")
            .alias("change"),

        col("changePercent")
            .cast("decimal(18,6)")
            .alias("change_percent"),

        col("vwap")
            .cast("decimal(18,4)")
            .alias("vwap"),

        current_timestamp()
            .alias("_ingestion_timestamp"),

        col("_metadata.file_path")
            .alias("_source_file"),
    )
)

In [0]:
df_bronze.printSchema()

In [0]:
print("Raw records:", df_raw.count())
print("Bronze records:", df_bronze.count())

### Write to Bronze Table

In [0]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("fintech.bronze.stock_prices")
)